# Práctica 3 — Modelo final con scikit-learn (Profesor)

## Flujo reproducible y sin fuga

Esta plantilla deja el test fuera de toda decisión. La imputación, codificación y escalado viven dentro del pipeline; `GridSearchCV` aprende y compara exclusivamente con `X_train` e `y_train`. No contiene resultados ejecutados porque dependen de la versión concreta del dataset.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
def find_project_root() -> Path:
    """Locate the repository from the current Jupyter working directory."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pixi.toml").is_file() and (candidate / "03-machine-learning").is_dir():
            return candidate
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del repositorio. Abre el notebook dentro del repositorio PIA."
    )


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "03-machine-learning/03-practicas/midterm-capstone/Peliculas/Practica_Peliculas_OK/data/movies.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"No se encuentra {DATA_PATH}. Solicita el movies.csv validado y guárdalo en esa ruta."
    )

df = pd.read_csv(DATA_PATH)
TARGET = "rating_high"
DROP_COLUMNS = []  # Completar con IDs o variables no disponibles al predecir.

X = df.drop(columns=[TARGET, *DROP_COLUMNS])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

## Pipeline ajustado sobre entrenamiento

Las listas de columnas se derivan de `X_train`. Esto evita que incluso decisiones de preprocesamiento se apoyen en el test. En cada fold, los imputadores, el codificador y el escalador se ajustan con los datos de entrenamiento de ese fold.


In [ ]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=42, class_weight="balanced")),
    ])


## Selección y ajuste solo sobre entrenamiento

F1 mantiene coherencia con P2 para una clasificación potencialmente desbalanceada. La cuadrícula es didáctica y deliberadamente acotada; una ampliación debe justificarse sin abrir el test.


In [ ]:
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2, 5],
}
search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    refit=True,
)
search.fit(X_train, y_train)

cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
display(cv_results[["rank_test_score", "mean_test_score", "std_test_score", "params"]].head())
print("Mejor F1 media de CV:", search.best_score_)
print("Hiperparámetros:", search.best_params_)
best_pipeline = search.best_estimator_


## Evaluación final: una sola vez sobre test

La decisión está cerrada antes de esta celda. Si el resultado obliga a una nueva hipótesis, no se reutiliza el test: se formula la iteración con entrenamiento/CV y se reserva una evaluación nueva.


In [ ]:
y_pred = best_pipeline.predict(X_test)

final_metrics = pd.Series(
    {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    }
)
display(final_metrics)
print(classification_report(y_test, y_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()


In [ ]:
error_analysis = X_test.copy()
error_analysis["actual"] = y_test
error_analysis["predicted"] = y_pred
errors = error_analysis.loc[error_analysis["actual"] != error_analysis["predicted"]]
errors.head(10)


## Revisión docente

Comprobar que el notebook entregado incluye la separación 80/20 con semilla y estratificación, pipeline completo, CV=5 limitada a entrenamiento, resultados de CV, una sola evaluación del test y análisis de errores. Las métricas no se preestablecen: se generan al ejecutar con el dataset elegido.


## Anexo GPU (opcional)

Una variante GPU debe preservar el mismo contrato de separación y evaluación.


In [ ]:
# from cuml.ensemble import RandomForestClassifier as RF_GPU
